In [13]:
import os
import random
import sys
import warnings
from pathlib import Path
import optuna
import mlflow
import time

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.models.har_wrapper import HARWrapper
from src.eval.scorer import VolatilityScorer
from src.trainer.walk_forward import WalkForwardTrainer
from src.utils import trainer_toolkit as tt
from src.utils import ts_toolkit as tst
from src.utils import test_toolkit as ts

warnings.filterwarnings("ignore")

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

mlflow.set_tracking_uri("sqlite:///mlflow.db")

OUT_PATH = PROJECT_ROOT / "data" / "processed"
RESULTS_PATH = PROJECT_ROOT / "results" / "har"
MODELS_PATH = PROJECT_ROOT / "models"

RESULTS_PATH.mkdir(parents=True, exist_ok=True)
MODELS_PATH.mkdir(parents=True, exist_ok=True)

df_train = pd.read_parquet(OUT_PATH / "brent_train.parquet")
y_train = df_train["log_returns"].values.astype("float64")
rs_train = df_train["rs_daily"].values.astype("float64")

df_test = pd.read_parquet(OUT_PATH / "brent_test.parquet")
y_test = df_test["log_returns"].values.astype("float64")
rs_test = df_test["rs_daily"].values.astype("float64")

assert not np.isnan(y_train).any() and not np.isnan(rs_train).any(), "NaNs en train"
assert not np.isnan(y_test).any() and not np.isnan(rs_test).any(), "NaNs en test"
assert df_train.index.max() < df_test.index.min(), "Solapamiento temporal train/test"

print("=" * 70)
print(f"Train: {len(y_train)} obs  ({df_train.index[0].date()} -> {df_train.index[-1].date()})")
print(f"Test:  {len(y_test)} obs  ({df_test.index[0].date()} -> {df_test.index[-1].date()})")
print("=" * 70)

Train: 2171 obs  (2014-01-03 -> 2022-11-03)
Test:  953 obs  (2022-11-07 -> 2026-08-21)


:::{admonition} Resumen
:class: info
- El protocolo Walk-Forward seleccionó al HAR con ventanas de 7 días (semanal) y 22 días (mensual) como la mejor configuración.
- El modelo validó empíricamente la Hipótesis de Mercado Heterogéneo de Corsi (2009), demostrando que la memoria de largo plazo del Brent se agota en el mes bursátil y ajustándose al ciclo calendario de 7 días.
- En contraste con el entrenamiento, los residuos fuera de muestra se ajustaron a una t-Student ($\nu \approx 6.035$) y superaron las pruebas de autocorrelación y dependencia no lineal, confirmando su eficacia en regímenes estables.
- La capacidad predictiva del modelo fuera de la muestra de entrenamiento fue similar a la del EGARCH, pero inferior a la del GARCH.
- A pesar de su parsimonia, la estructura de suavizado de los promedios móviles subestimó la magnitud del pico extremo de 2026, evidenciando una rigidez estructural ante cambios abruptos de régimen y una alta sensibilidad a la asimetría del QLIKE.
:::

## Introducción

El modelo GARCH y sus extensiones (como el EGARCH) capturan la persistencia de la volatilidad a través de los rezagos de la varianza condicional y de los choques recientes. Sin embargo, la literatura empírica ha demostrado que la volatilidad de los activos financieros exhibe una estructura de memoria larga que los modelos GARCH tradicionales no logran capturar eficientemente sin imponer restricciones de estacionariedad que rozan la integración fraccional. 

Para abordar esta limitación, Corsi (2009) propone el modelo Heterogeneous Autoregressive (HAR). Basado en la Hipótesis de Mercado Heterogéneo (Haugen, 1995; Müller et al., 1997), el HAR asume que el mercado está compuesto por agentes que operan a distintos horizontes temporales (intradía, semanal y mensual), lo que genera una estructura de volatilidad con múltiples frecuencias. Esta sección presenta, estima y evalúa el modelo HAR bajo el mismo protocolo experimental empleado para los modelos GARCH, con el fin de garantizar la comparabilidad de sus resultados.

### Modelo HAR Original (HAR-RV)

En su formulación original, el HAR modela la volatilidad realizada (RV) como una función lineal de sus valores pasados agregados en tres escalas temporales:

$$
RV_t = c + \beta_d RV_{t-1}^{(d)} + \beta_w RV_{t-1}^{(w)} + \beta_m RV_{t-1}^{(m)} + \varepsilon_t
$$

Donde:
- $RV_{t-1}^{(d)} = RV_{t-1}$ representa la volatilidad realizada diaria.
- $RV_{t-1}^{(w)} = \frac{1}{5}\sum_{i=1}^{5} RV_{t-i}$ representa el promedio de la volatilidad de la última semana (5 días hábiles).
- $RV_{t-1}^{(m)} = \frac{1}{22}\sum_{i=1}^{22} RV_{t-i}$ representa el promedio de la volatilidad del último mes (22 días hábiles).

Los coeficientes $\beta_d$, $\beta_w$ y $\beta_m$ capturan la influencia de los agentes de corto, mediano y largo plazo, respectivamente. Este modelo aproxima la memoria larga mediante una estructura autorregresiva simple, requiriendo únicamente la estimación de tres parámetros.

### Adaptación Metodológica: HAR sobre Retornos (HAR-RS)

Para mantener la consistencia metodológica estricta exigida por el benchmark de este trabajo, el modelo HAR original debe ser adaptado. Los modelos paramétricos (GARCH, EGARCH) y los modelos de Deep Learning (GRU, LSTM, TCN) empleados en este estudio tienen dos características en común: (i) reciben exclusivamente la serie de retornos históricos como *input*, y (ii) su variable objetivo (*target*) es la varianza de Rogers-Satchell (RS).

Por lo tanto, se descarta el uso de la volatilidad realizada intradía ($RV$) como insumo, y se adapta el HAR para operar bajo las mismas condiciones. El modelo, denotado como HAR-RS, se especifica de la siguiente manera:

$$
\widehat{RS}_t = c + \beta_d r_{t-1}^2 + \beta_w \left(\frac{1}{L_w}\sum_{i=1}^{L_w} r_{t-i}^2\right) + \beta_m \left(\frac{1}{L_m}\sum_{i=1}^{L_m} r_{t-i}^2\right) + \varepsilon_t
$$

Donde $r_{t-i}$ representa el retorno logarítmico diario. Al elevar los retornos al cuadrado ($r_{t-i}^2$), el modelo utiliza el cuadrado de los retornos como un proxy de la volatilidad pasada. Bajo el supuesto de que la media condicional de los retornos es cero ($\mu_t = 0$), se cumple que $\mathbb{E}_{t-1}[r_t^2] = \sigma_t^2$, lo que valida teóricamente el uso del retorno al cuadrado como estimador insesgado de la varianza condicional latente. La variable dependiente ($\widehat{RS}_t$) es la varianza de Rogers-Satchell, garantizando que el modelo predice exactamente el mismo *target* que los modelos de Deep Learning.

Si bien la formulación clásica de Corsi (2009) define $L_w=5$ y $L_m=22$, en esta implementación los órdenes de los rezagos semanal ($L_w$) y mensual ($L_m$) se tratan como hiperparámetros sujetos a optimización bayesiana (Optuna), permitiendo al modelo explorar ventanas como 7 y 30 días. Esto permite adaptar la estructura de memoria a las particularidades del mercado del petróleo Brent. Asimismo, para prevenir estrictamente cualquier fuga de información (*data leakage*), todas las agregaciones se calculan aplicando un rezago de un periodo (`shift(1)`) sobre los retornos, garantizando que la predicción en $t$ solo utilice información disponible hasta $t-1$.

### Estimación y Supuestos

La estimación de los parámetros del modelo HAR-RS se realiza mediante Mínimos Cuadrados Ordinarios (OLS), dado que el modelo es lineal en sus parámetros. El uso de OLS sigue siendo válido y es el estándar en la literatura (Corsi, 2009) bajo los siguientes supuestos fundamentales:

1. **Linealidad y Correcta Especificación:** La relación entre la varianza RS y los proxies de volatilidad pasada es lineal en los parámetros.
2. **Media Condicional del Error Cero:** El término de error $\varepsilon_t$ satisface $\mathbb{E}[\varepsilon_t \mid \mathcal{F}_{t-1}] = 0$, lo que implica que el modelo no presenta sesgo sistemático.
3. **No Multicolinealidad Perfecta:** Aunque las medias móviles diaria, semanal y mensual están altamente correlacionadas, no son perfectamente colineales, permitiendo la inversión de la matriz de diseño.
4. **Estacionariedad:** La serie de retornos al cuadrado debe ser estacionaria para que los coeficientes estimados sean consistentes y las predicciones no diverjan. Aunque los retornos al cuadrado suelen exhibir autocorrelación serial (induciendo heterocedasticidad condicional en los residuos), OLS sigue proporcionando estimadores insesgados y consistentes para el pronóstico.

Dado que las series financieras suelen presentar heterocedasticidad condicional y autocorrelación en los residuos, la inferencia estadística sobre los coeficientes se realiza reportando errores estándar consistentes con heterocedasticidad y autocorrelación (HAC), siguiendo la corrección de Newey-West con un rezago igual a $L_m$. Si bien esta corrección no altera los coeficientes puntuales utilizados para el pronóstico, garantiza que los p-valores y los intervalos de confianza reportados en la tabla de resultados sean estadísticamente robustos.

El modelo se implementa computacionalmente utilizando la librería `statsmodels` de Python, mediante el estimador `OLS`. El pronóstico se genera de forma recursiva (expansiva) bajo el protocolo de validación *walk-forward*, re-estimando los parámetros en cada paso con la información disponible hasta el momento.

## Entrenamiento

Para la selección de hiperparámetros del modelo HAR, se aplica el mismo protocolo experimental detallado en las secciones anteriores. Se ejecuta una réplica bajo el esquema Walk-Forward, utilizando optimización bayesiana (Optuna) para explorar las longitudes de las ventanas temporales semanal ($L_w$) y mensual ($L_m$). Siguiendo la formulación original de Corsi (2009) y permitiendo cierta flexibilidad para adaptarse al mercado del Brent, $L_w$ explora los valores $\{5, 7\}$ días, mientras que $L_m$ explora $\{22, 30\}$ días. 

Se mantienen idénticos los criterios de control computacional (límite de 50 evaluaciones y parada temprana tras 15 intentos). En la práctica, dado que el HAR es un modelo lineal y el espacio de búsqueda se reduce a solo cuatro combinaciones posibles, la parada temprana se activa rápidamente al agotarse las combinaciones válidas.

A diferencia de los modelos GARCH y EGARCH, donde se aplica un factor de escalado a los retornos (multiplicación por 100) para garantizar la estabilidad numérica del optimizador de máxima verosimilitud (MLE), el HAR se estima mediante Mínimos Cuadrados Ordinarios (OLS). Dado que el OLS es algebraicamente invariante a transformaciones de escala, no se aplicó ningún factor de escalado ni a los retornos ni a la varianza objetivo. Esto no solo evita pasos innecesarios, sino que mantiene la interpretabilidad directa de los coeficientes estimados. La naturaleza determinista del OLS asegura que no existan fallos de convergencia, a diferencia de lo observado en algunas iteraciones del GARCH.


In [4]:
def har_search_space(trial: optuna.Trial) -> dict:
    return {
        "lag_weekly": trial.suggest_categorical("lag_weekly", [5, 7]),
        "lag_monthly": trial.suggest_categorical("lag_monthly", [22, 30]),
        "seed": 42
    }

exec_id = 6
current_warmup = 1200 + (exec_id * 10)

exec_dir = RESULTS_PATH / f"exec_{exec_id}"
exec_dir.mkdir(parents=True, exist_ok=True)

trainer = WalkForwardTrainer(
    model_class=HARWrapper,
    scorer=VolatilityScorer(),
    output_dir=exec_dir,
    n_splits=5, 
    warmup=current_warmup,
    embargo=1, 
    n_trials=50,
    suggest_params_fn=har_search_space,
    early_stop_rounds=15
)

trainer.run(y_train, rs_train)

2026/11/06 15:52:04 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/11/06 15:52:04 INFO mlflow.store.db.utils: Updating database tables
2026/11/06 15:52:14 INFO mlflow.tracking.fluent: Experiment with name 'Brent_Volatility_Benchmark' does not exist. Creating a new experiment.


  0%|          | 0/50 [00:00<?, ?it/s]

La búsqueda identificó como configuración óptima al HAR con ventanas de 7 y 22 días ($L_w=7$, $L_m=22$), lo que evidencia una rápida exploración del reducido espacio de hiperparámetros. El mecanismo de poda (*Pruning*) no descartó ninguna configuración (0 podados), lo que indica que la estimación por Mínimos Cuadrados Ordinarios (OLS) garantiza la convergencia analítica desde la primera partición, evitando cualquier descarte por inestabilidad numérica.

El entrenamiento completo bajo el esquema Walk-Forward totalizó 0.92 segundos, con un tiempo medio de 0.046 segundos por partición. Este costo mantiene (y supera) la tendencia de alta eficiencia computacional de los modelos paramétricos, permitiendo su estimación y actualización diaria en entornos de producción de forma rápida sin requerir hardware especializado.

In [9]:
resumen_data = []

exec_id = 6
exec_dir = RESULTS_PATH / f"exec_{exec_id}"

tiempo_total_rep = 0.0
tiempo_promedio_fold = 0.0
num_folds = 0
total_trials, completos, podados, fallidos = 0, 0, 0, 0
best_params_str = "-"
mejor_qlike = "-"

csv_files = list(exec_dir.glob("*_results.csv"))
if csv_files:
    df_csv = pd.read_csv(csv_files[0])
    tiempo_total_rep = df_csv["train_time_sec"].sum()
    tiempo_promedio_fold = df_csv["train_time_sec"].mean()
    num_folds = len(df_csv)

db_path = exec_dir / "optuna_study.db"
if db_path.exists():
    try:
        storage_url = f"sqlite:///{db_path.resolve()}"
        study = optuna.load_study(study_name="HARWrapper_study", storage=storage_url)

        total_trials = len(study.trials)
        completos = len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])
        podados  = len([t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED])
        fallidos = len([t for t in study.trials if t.state == optuna.trial.TrialState.FAIL])

        mejor_qlike = round(study.best_value, 4)
        
        w_val = study.best_params.get('lag_weekly', '?')
        m_val = study.best_params.get('lag_monthly', '?')
        best_params_str = f"({w_val}, {m_val})"

    except Exception as e:
        print(f"Error leyendo BD en {exec_dir}: {e}")

resumen_data.append({
    "Total Trials": total_trials,
    "Completos": completos,
    "Podados": podados,
    "Mejor Config (Lw, Lm)": best_params_str,
    "Mejor QLIKE": mejor_qlike,
    "Tiempo Total (s)": round(tiempo_total_rep, 2),
    "Tiempo por Fold (s)": round(tiempo_promedio_fold, 3)
})

df_search_params = pd.DataFrame(resumen_data)
tt.display_styled_df(df=df_search_params, decimals=4, align_left_first=False)

Total Trials,Completos,Podados,"Mejor Config (Lw, Lm)",Mejor QLIKE,Tiempo Total (s),Tiempo por Fold (s)
18,18,0,"(7, 22)",0.7287,0.9200,0.0460


El análisis de las cuatro configuraciones evaluadas del HAR evidenció una dicotomía en el desempeño del modelo. La configuración de 7 días semanales y 22 días mensuales se seleccionó como la óptima, al registrar el menor QLIKE (0.729). La configuración de 5 y 22 días presentó un rendimiento estadísticamente equivalente (0.733), dado que la diferencia entre ambas (0.004) resultó inferior a la variabilidad entre folds ($\sigma \approx 0.21$). A pesar de esta equivalencia estadística, la selección de la ventana de 7 días se fundamentó en su carácter de mínimo empírico y en consideraciones del mercado: el petróleo Brent es susceptible a anuncios geopolíticos y decisiones de la OPEP que suelen ocurrir durante los fines de semana, por lo que una ventana calendario de 7 días capturó dicha continuidad temporal, a diferencia de los 5 días bursátiles de la formulación original. 

In [10]:
exec_id = 6
exec_dir = RESULTS_PATH / f"exec_{exec_id}"

df_har = pd.read_csv(exec_dir / "HARWrapper_results.csv")
cols_agrupar = ["lag_weekly", "lag_monthly"]
df_agg = df_har.groupby(cols_agrupar)["qlike"].agg(["mean", "std"]).reset_index()

df_agg["QLIKE"] = df_agg.apply(lambda x: f"{x['mean']:.3f} ± {x['std']:.3f}", axis=1)
df_top5 = df_agg.sort_values(by="mean").head(5)

df_top5 = df_top5.rename(columns={
    "lag_weekly": "Lag Semanal (Lw)",
    "lag_monthly": "Lag Mensual (Lm)"
})

df_final = df_top5[["Lag Semanal (Lw)", "Lag Mensual (Lm)", "QLIKE"]]
tt.display_styled_df(df=df_final, align_left_first=False, decimals=3)

Lag Semanal (Lw),Lag Mensual (Lm),QLIKE
7,22,0.729 ± 0.212
5,22,0.733 ± 0.214
7,30,0.932 ± 0.361
5,30,0.945 ± 0.369


Por otro lado, extender la ventana mensual a 30 días incrementó el QLIKE por encima de 0.93. Esto indicó que la memoria de largo plazo del activo se agotó en los 22 días hábiles, incorporando información de regímenes anteriores que no aportó capacidad predictiva. En consecuencia, los resultados validaron la estructura de la Hipótesis de Mercado Heterogéneo (Corsi, 2009) en cuanto a la escala mensual, y permitieron ajustar la escala semanal a las particularidades del mercado del Brent.

In [2]:
best_model = HARWrapper(lag_weekly=7, lag_monthly=22)
best_model.fit(y_train, rs_train)

La estimación de los coeficientes del modelo HAR evidenció que las tres componentes temporales presentaron un impacto positivo y significativo sobre la volatilidad. Dado que las variables compartieron la misma escala, la magnitud de sus coeficientes permitió comparar directamente su influencia: la componente mensual registró el mayor impacto sobre la varianza (0.4027), seguida por la diaria (0.3478) y, en menor medida, por la semanal (0.0989). Esto indicó que los agentes de largo plazo dominaron la formación de la volatilidad en el mercado del Brent durante el periodo de estudio. 

In [3]:
df_params_boot = best_model.get_parameter_table(
    B=2000, decimals=4, block_size=22, verbose=True
)

df_params_boot = df_params_boot.rename(columns={
    "Parameter": "Parámetro",
    "Coef": "Coeficiente",
    "Std. Error": "Error Estándar",
    "P-value": "p-valor",
    "CI Lower": "IC Inferior (95%)",
    "CI Upper": "IC Superior (95%)"
})

tt.display_styled_df(
    df=df_params_boot,
    decimals=4,
    p_value_cols=["p-valor"]
)

Block Bootstrap:   0%|          | 0/2000 [00:00<?, ?it/s]

Parámetro,Coeficiente,Error Estándar,p-valor,IC Inferior (95%),IC Superior (95%)
const,0.0000,0.0000,.418,-0.0001,0.0001
RV_d,0.3478,0.0140,< .001,0.3308,0.3665
RV_w,0.0989,0.0428,.021,0.0301,0.1812
RV_m,0.4027,0.0506,< .001,0.3223,0.5153


La suma de los tres coeficientes (0.8494) resultó inferior a la unidad. En el marco de los modelos autorregresivos, este resultado implicó que los choques de volatilidad presentaron una alta persistencia, pero tendieron a disiparse en el largo plazo (reversión a la media). Finalmente, los intervalos de confianza al 95 %, calculados mediante bootstrapping de bloques, no incluyeron el cero en ninguna de las tres componentes, confirmando la robustez de los estimadores.

A partir de las varianzas predichas y los retornos reales del conjunto de entrenamiento, se construyeron los residuos estandarizados en muestra mediante la siguiente expresión:

$$
z_t = \frac{y_t}{\sqrt{\hat{\sigma}_t^2}}
$$

Para evaluar la estructura de estos residuos, se aplicaron pruebas de diagnóstico. A diferencia de los modelos GARCH y EGARCH, las pruebas de Ljung-Box (autocorrelación), ARCH-LM (heterocedasticidad) y BDS (dependencia no lineal) rechazaron la hipótesis nula (*p* < .05). Lo anterior indicó que la estructura lineal del HAR no logró capturar la dependencia temporal, los efectos condicionales ni las dinámicas no lineales del mercado.

In [4]:
residuos_har = best_model.model_fit.resid
std_residuals = residuos_har / residuos_har.std()

df_diag = tt.run_model_diagnostics(
    std_residuals=std_residuals,
    lags=(10, 20, 30),
    p_adjust="bonferroni",
    alpha=0.05,
    english=False,
    column_titles=(
        "Prueba Estadística", "Estadístico",
        "p-valor", "p-adj", "Conclusión (α = 0.05)")
)

df_display = df_diag.copy()

df_display["Estadístico"] = df_display["Estadístico"].apply(
    lambda x: f"{x:.4f}" if pd.notna(x) else "--")

def _format_p(x):
    if pd.isna(x): return "--"
    if x < 0.001: return "< .001"
    formatted = f"{x:.3f}"
    return formatted[1:] if formatted.startswith("0.") else formatted

df_display["p-valor"] = df_display["p-valor"].apply(_format_p)
df_display["p-adj"] = df_display["p-adj"].apply(_format_p)

tt.display_styled_df(df=df_display, p_value_cols=[])

Prueba Estadística,Estadístico,p-valor,p-adj,Conclusión (α = 0.05)
Ljung-Box (Lag 10),131.4296,< .001,< .001,Autocorrelación Significativa
Ljung-Box (Lag 20),232.5019,< .001,< .001,Autocorrelación Significativa
Ljung-Box (Lag 30),389.2941,< .001,< .001,Autocorrelación Significativa
ARCH-LM (Lag 10),260.7706,< .001,< .001,Efectos ARCH Significativos
ARCH-LM (Lag 20),278.3568,< .001,< .001,Efectos ARCH Significativos
ARCH-LM (Lag 30),392.9892,< .001,< .001,Efectos ARCH Significativos
Jarque-Bera,18948508.6530,< .001,--,Residuos No Normales
Lilliefors,0.3059,< .001,--,Residuos No Normales
BDS (Dim 2),14.9214,< .001,--,Estructura No Lineal Significativa


Esta limitación del modelo HAR para capturar la dinámica temporal puede relacionarse con la rigidez de sus promedios móviles o se puede deber al ruido de su proxy de entrada. Se planteó como hipótesis que, al usar pesos fijos (1/5 y 1/22), el HAR suaviza en exceso los choques y no logra el ajuste dinámico que sí tiene el GARCH mediante la actualización recursiva de la varianza. El uso de retornos al cuadrado en lugar de volatilidad realizada pudo introducir ruido de medición, el cual se refleja como una aparente dependencia no lineal en el test BDS.

El análisis de la distribución de los residuos evidenció que la estructura del HAR no capturó adecuadamente la forma de los choques. La inspección visual del histograma reveló una distribución altamente leptocúrtica, caracterizada por una concentración extrema en el centro y colas muy pesadas. Esta falta de ajuste a las distribuciones teóricas se confirmó estadísticamente: los residuos no siguieron una distribución normal (Lilliefors: $p < .001$) ni una t-Student (Kolmogorov-Smirnov: $p < .001$; Cramér-von Mises: $p < .001$). En conjunto, estos resultados evidenciaron que la estructura del HAR no capturó la curtosis empírica de los errores, limitando su capacidad para representar la verdadera forma de las innovaciones del mercado.

In [7]:
_, _ = tst.plot_distribution_fit(
    pd.Series(std_residuals),
    dist="none",
    titles=("Distribución de Residuos", "Gráfico Q-Q (normal)"),
    xlab="Residuo estandarizado",
    ylab_left="Densidad",
    xlab_right="Cuantiles teóricos",
    ylab_right="Cuantiles muestrales",
    ylim_left=(0, 2),
    legend_loc="upper left"
)

La inspección visual del ajuste en la muestra de entrenamiento mostró que la estructura del HAR logró seguir adecuadamente la tendencia general y los ciclos temporales de la volatilidad, sin presentar desfases evidentes en el tiempo. No obstante, el modelo exhibió un marcado efecto de suavizado sobre la serie. Como se evidencia en el zoom de la crisis por COVID-19, la volatilidad estimada por el HAR no alcanzó la magnitud de los picos extremos de la volatilidad realizada. Este subajuste en los eventos de mayor intensidad se relaciona directamente con el comportamiento de los residuos analizado previamente: al no capturar la magnitud de los shocks extremos, los errores conservaron las colas pesadas y la estructura leptocúrtica observada en el histograma.

In [8]:
pred_varianza_har = best_model.model_fit.fittedvalues

lag = best_model.lag_monthly
rs_volatilidad_real = np.sqrt(rs_train[lag:])
pred_volatilidad_har = np.sqrt(pred_varianza_har)
fechas_alineadas = df_train.index[lag:]

mask_crisis = (fechas_alineadas >= "2020-01-01") & (fechas_alineadas <= "2020-06-30")

ts.plot_volatility_fit(
    dates=fechas_alineadas,
    rs_variance=rs_volatilidad_real,      
    pred_variance=pred_volatilidad_har,
    title="Ajuste dentro la Muestra de Entrenamiento (HAR 7,22)", 
    ylab="Volatilidad",              
    label_realized=r"Volatilidad Realizada $\left(\sigma_R\right)$", 
    label_pred=r"Volatilidad HAR $\left(\hat{\sigma}_t\right)$",
    ylim=(-0.01, 0.35), 
    legend_bbox=(0.01, 0.98), 
    zoom_slice=mask_crisis, 
    zoom_pad_frac=0.04,
    zoom_title="Zoom: Crisis COVID-19",
    zoom_bbox=(0.18, 0.35, 0.40, 0.40),
    title_size=14, label_size=14, tick_size=12, legend_size=14
)

## Evaluación

La evaluación del HAR sigue el mismo protocolo experimental detallado en las secciones anteriores. El rendimiento se contrasta comparando el ajuste sobre los datos observados con la capacidad predictiva sobre datos no vistos. 

Para diagnosticar la presencia de sobreajuste (*overfitting*) o subajuste (*underfitting*), se mantiene el criterio de la diferencia absoluta entre las métricas de prueba y entrenamiento:

$$
\Delta_{abs} = M_{test} - M_{train}
$$

Al igual que en los modelos anteriores, un valor de $|\Delta_{abs}|$ que supere el umbral de tolerancia de 0.05 evidencia falta de generalización. 

Las predicciones fuera de muestra se generan ejecutando el método `forecast`, el cual aplica un esquema de ventana expansiva (*rolling forward*) que reestima los coeficientes OLS en cada paso para garantizar la ausencia de fuga de información. A diferencia de los modelos GARCH y EGARCH, el HAR no requiere factores de escalado numérico para su estabilidad, por lo que las varianzas pronosticadas se comparan directamente con la volatilidad realizada en su escala original.

In [14]:
start_time = time.time()

preds_test = best_model.forecast(y_seq=y_test, rs_seq=rs_test)
end_time = time.time()
tiempo_evaluacion = end_time - start_time


df_preds = pd.DataFrame({
    "date": df_test.index, 
    "rs_true": rs_test, 
    "y_pred": preds_test
})

output_path = RESULTS_PATH / "best_har_predictions.csv"
df_preds.to_csv(output_path, index=False)

La generación de pronósticos fuera de muestra mediante el esquema de ventana expansiva y la reestimación del OLS en cada paso tuvo una duración total de 20.26 segundos. Esto equivalió a un tiempo medio de aproximadamente 0.021 segundos por predicción (para las 953 observaciones del conjunto de prueba). Este costo es consistente con la alta eficiencia computacional de los modelos paramétricos, demostrando que la actualización iterativa de una regresión lineal es más rápida en comparación con el reentrenamiento de arquitecturas de Deep Learning.

El análisis de los residuos fuera de muestra presentó un comportamiento parcialmente distinto al observado durante el entrenamiento. Las pruebas de Ljung-Box no rechazaron la hipótesis nula de autocorrelación ($p\text{-adj} > .05$), lo que indicó que el modelo logró capturar la dependencia temporal lineal en datos no vistos. No obstante, la prueba ARCH-LM detectó efectos ARCH significativos en todos los rezagos evaluados ($p\text{-adj} < .001$), evidenciando que el HAR no absorbió por completo el agrupamiento de la volatilidad (*volatility clustering*) de los errores. La prueba BDS no rechazó la hipótesis nula de ausencia de estructura no lineal ($p = .219$), contrastando con lo observado en la muestra de entrenamiento.

In [15]:
std_residuals_oos = y_test / np.sqrt(preds_test)
std_residuals_oos = std_residuals_oos[np.isfinite(std_residuals_oos)]

df_diag = tt.run_model_diagnostics(
    std_residuals=std_residuals_oos,
    lags=(10, 20, 30),
    p_adjust="bonferroni",
    alpha=0.05,
    english=False,
    column_titles=(
        "Prueba Estadística", "Estadístico",
        "p-valor", "p-adj", "Conclusión (α = 0.05)")
)

df_display = df_diag.copy()

df_display["Estadístico"] = df_display["Estadístico"].apply(
    lambda x: f"{x:.4f}" if pd.notna(x) else "--")

def _format_p(x):
    if pd.isna(x): return "--"
    if x < 0.001: return "< .001"
    formatted = f"{x:.3f}"
    return formatted[1:] if formatted.startswith("0.") else formatted

df_display["p-valor"] = df_display["p-valor"].apply(_format_p)
df_display["p-adj"] = df_display["p-adj"].apply(_format_p)

tt.display_styled_df(df=df_display, p_value_cols=[])

Prueba Estadística,Estadístico,p-valor,p-adj,Conclusión (α = 0.05)
Ljung-Box (Lag 10),14.9558,.134,.401,Sin Autocorrelación
Ljung-Box (Lag 20),26.8370,.140,.420,Sin Autocorrelación
Ljung-Box (Lag 30),29.5316,.490,1.000,Sin Autocorrelación
ARCH-LM (Lag 10),66.1934,< .001,< .001,Efectos ARCH Significativos
ARCH-LM (Lag 20),69.9507,< .001,< .001,Efectos ARCH Significativos
ARCH-LM (Lag 30),71.0748,< .001,< .001,Efectos ARCH Significativos
Jarque-Bera,289.4597,< .001,--,Residuos No Normales
Lilliefors,0.0452,< .001,--,Residuos No Normales
BDS (Dim 2),1.2304,.219,--,Sin Estructura No Lineal


Esta discrepancia en la prueba BDS se explica por la sensibilidad del HAR a la presencia de choques extremos, derivada de su estructura de suavizado. El conjunto de entrenamiento incluyó la crisis del COVID-19, un evento de volatilidad abrupta y no lineal que los promedios móviles fijos (7 y 22 días) no pudieron capturar, dejando dependencias no lineales en los residuos. Por el contrario, el conjunto de prueba presentó un régimen de volatilidad más estable y sin choques estructurales severos. En este contexto de mayor normalidad, la estructura de promedios del HAR fue suficiente para explicar la dinámica de los retornos sin dejar patrones no lineales significativos, confirmando que el modelo es adecuado para regímenes estables pero estructuralmente limitado ante eventos extremos.

El análisis de la distribución de los residuos fuera de muestra reveló un ajuste adecuado a la distribución t-Student (Kolmogorov-Smirnov: $p = 0.537$; Cramér-von Mises: $p = 0.630$). Esto fue confirmado por la prueba de razón de verosimilitud (LRT), la cual evidenció que la distribución t-Student se ajustó significativamente mejor que la distribución gaussiana ($p < .001$). Es fundamental destacar el cambio en el parámetro $\nu$: mientras que en el entrenamiento este colapsó a un valor de $1.1781$, en el conjunto de prueba se situó en $6.0349$. Este incremento en $\nu$ fue consistente con un régimen de volatilidad más estable.

La inspección visual corroboró estos resultados estadísticos. El histograma de los residuos mostró un ajuste preciso a la curva teórica de la t-Student, y el gráfico Q-Q evidenció que los cuantiles muestrales se alinearon con la línea diagonal de referencia en la mayor parte de la distribución. En conjunto, esta evidencia indicó que el HAR capturó la forma de los choques de manera adecuada en el conjunto de prueba. La diferencia en $\nu$ entre ambas muestras demostró la sensibilidad del modelo a los eventos extremos, confirmando que su estructura lineal es efectiva en periodos de estabilidad, pero limitada ante eventos de alta volatilidad como la crisis del COVID-19.

In [23]:
_, _ = tst.plot_distribution_fit(
    pd.Series(std_residuals_oos),
    dist="t",
    titles=("Distribución de Residuos", "Gráfico Q-Q (normal)"),
    xlab="Residuo estandarizado",
    ylab_left="Densidad",
    xlab_right="Cuantiles teóricos",
    ylab_right="Cuantiles muestrales",
    ylim_left=(0, 0.6),
    legend_loc="upper left"
)

El modelo HAR no cumplió con los criterios de adecuación pre-fijados para la métrica principal. La pérdida QLIKE superó el umbral de tolerancia absoluto establecido ($\Delta = +0.0770 > 0.05$), lo que evidenció una disminución de la capacidad predictiva fuera de la muestra. Cabe destacar que el valor en prueba del HAR (0.5076) resultó similar al registrado por el modelo EGARCH (0.5068). Este incremento en el QLIKE, en contraste con las variaciones marginales de las métricas puntuales (MSE, RMSE, MAE) y probabilísticas (CRPS y Pinball), se explica por la asimetría de la función de pérdida. El QLIKE penaliza severamente la subestimación de la varianza, por lo que los errores cometidos durante los días de mayor turbulencia recibieron un castigo proporcionalmente mayor en el conjunto de prueba.

Las métricas ajustadas por heterocedasticidad superaron el umbral establecido, con incrementos de $+3.0418$ en el HMSE y $+0.0753$ en el HMAE. Este resultado es consistente con la definición del HMSE, el cual pondera los errores por el inverso de la volatilidad. Esto magnificó el error de predicción durante los días de mayor turbulencia en el conjunto de prueba.

In [16]:
preds_train_var = best_model.model_fit.fittedvalues
preds_test_var = df_preds["y_pred"].values

lag = best_model.lag_monthly
rs_train_aligned = rs_train[lag:]

ts.display_train_test_comparison(
    rs_train=rs_train_aligned,
    preds_train=preds_train_var,
    rs_test=rs_test,
    preds_test=preds_test_var,
    column_titles=(
        "Métrica", 
        "Entrenamiento", 
        "Prueba", 
        "Diferencia"
    )
)

Métrica,Entrenamiento,Prueba,Diferencia
QLIKE,0.4307,0.5076,+0.0770
CRPS,0.0128,0.0125,-0.0003
Pinball (τ=0.05),0.0027,0.0026,-0.0001
Pinball (τ=0.95),0.0014,0.0017,+0.0003
MSE,0.0000,0.0000,+0.0000
RMSE,0.0017,0.0022,+0.0006
MAE,0.0004,0.0005,+0.0000
HMSE,2.5360,5.5778,+3.0418
HMAE,0.7811,0.8565,+0.0753


La inspección visual de los pronósticos fuera de muestra corroboró el comportamiento descrito por las métricas. Durante los periodos de estabilidad, la volatilidad predicha por el HAR siguió de cerca a la volatilidad realizada, confirmando la capacidad del modelo para capturar la dinámica base del mercado. Sin embargo, en el pico de volatilidad registrado en 2026, la curva predicha se mantuvo por debajo de la volatilidad observada. Esta subestimación fue consecuencia directa del efecto de suavizado de los promedios móviles fijos. Al promediar los días previos, el modelo atenuó la magnitud del choque, presentando una reacción de menor amplitud. Esta evidencia visual explicó la divergencia entre las métricas de error simétricas y la pérdida asimétrica (QLIKE), indicando que la estructura del HAR se limita a describir regímenes estables y no logra dimensionar eventos de alta volatilidad.

In [29]:
rs_test_vol = np.sqrt(df_preds["rs_true"])
pred_test_vol = np.sqrt(df_preds["y_pred"])

mask_crisis = (df_preds["date"] >= "2026-02-01") & (df_preds["date"] <= "2026-06-30")

ts.plot_volatility_fit(
    dates=df_preds["date"],
    rs_variance=rs_test_vol,      
    pred_variance=pred_test_vol,
    title="Ajuste fuera de la Muestra de Entrenamiento (HAR 7, 22)",
    ylab="Volatilidad",              
    label_realized=r"Volatilidad Realizada $\left(\sigma_R\right)$", 
    label_pred=r"Volatilidad HAR $\left(\hat{\sigma}_t\right)$", 
    
    ylim=(-0.01, 0.35), 
    legend_bbox=(0.01, 0.98), 
    
    zoom_slice=mask_crisis, zoom_pad_frac=0.04,
    zoom_title="Zoom: Pico de Volatilidad (2026)",
    zoom_bbox=(0.25, 0.35, 0.40, 0.40),
    title_size=14, label_size=14, tick_size=12, legend_size=14
)

## Conclusiones

La evaluación del modelo HAR arrojó una pérdida QLIKE en el conjunto de prueba de 0.5076, un valor similar al del EGARCH (0.5068) pero superior al del GARCH (0.4097). La optimización seleccionó ventanas de 7 días semanales y 22 días mensuales, confirmando empíricamente la estructura de la Hipótesis de Mercado Heterogéneo de Corsi (2009) para el Brent: la memoria de largo plazo se agotó en el mes bursátil, mientras que la ventana semanal se ajustó al ciclo calendario del activo. Esto demostró que la dinámica principal de la volatilidad puede ser capturada mediante un modelo lineal altamente parsimonioso.

El modelo presentó un desempeño adecuado durante los periodos de estabilidad, pero evidenció un efecto de suavizado ante eventos extremos. Debido a la rigidez de sus promedios móviles, la volatilidad predicha no alcanzó la magnitud del pico registrado en 2026. Dado que la función QLIKE penaliza asimétricamente la subestimación de la varianza, esta falta de amplitud resultó en un incremento del error, el cual también se reflejó en las métricas ajustadas por heterocedasticidad (HMSE y HMAE). 

El análisis de los residuos confirmó que la estructura del HAR es adecuada para modelar regímenes de estabilidad, pero vulnerable a eventos extremos. La ausencia de autocorrelación y de dependencia no lineal en el conjunto de prueba, junto con un ajuste adecuado a una distribución t-Student con colas menos pesadas ($\nu \approx 6.035$), evidenció que el modelo logra explicar la dinámica del Brent cuando no existen choques severos. Este comportamiento contrastó con el entrenamiento, donde la crisis del COVID-19 rompió los supuestos lineales del modelo. Finalmente, a pesar de su alta eficiencia computacional (con tiempos de inferencia de milisegundos gracias al OLS), la ausencia de una actualización recursiva de la varianza impidió que el HAR igualara la capacidad de reacción dinámica del GARCH.